In [28]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm shap xlsxwriter


In [29]:
import sys
!{sys.executable} -m pip install -q imblearn

In [30]:
from imblearn.over_sampling import SMOTE

In [31]:
import os, re, time, json, warnings
from pathlib import Path
from typing import Dict, List

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import shap

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from imblearn.over_sampling import SMOTE

OUTPUT_DIR = Path("/content/outputs_v26_amplitude_h5_amplitude_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE   = 42
MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START    = "2000-01-01"
TEST_DATE      = "2022-01-01"

YF_CHUNK_SIZE          = 40
SLEEP_BETWEEN_CHUNKS   = 1.0
MIN_COLUMN_COVERAGE    = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365
ROLLING_QUANTILE_WINDOW  = 504

# Jours flat supprimés (|rendement| < 0.3%)
FLAT_THRESHOLD_ABS = 0.0   # conserve tous les jours

# Classes : q25/q75 calculés PAR RÉGIME sur le train
# (un mouvement "fort" dans CALM n'a pas la même amplitude que dans STRESS)
CLASS_QUANTILES = [0.25, 0.75]
CLASS_LABELS    = ["DOWN_FORT", "DOWN_FAIBLE", "UP_FAIBLE", "UP_FORT"]

# h=5j uniquement
HORIZONS_TO_TEST       = [5]
ALL_REGIMES            = ["CALM", "NORMAL", "STRESS", "GLOBAL"]

SHAP_PILOT_N_ESTIMATORS  = 150
SHAP_TOP_BASE_N          = 40
SHAP_TOP_FINAL_N         = 30
N_TOP_FOR_INTERACTIONS   = 20
INTERACTION_ROLLING_WINDOW = 20

MIN_N_FEATURES = 5
MAX_N_FEATURES = 15

np.random.seed(RANDOM_STATE)


In [32]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [33]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [34]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [35]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [36]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [37]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [38]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [39]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [40]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [41]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [42]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [43]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [44]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [45]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TBP']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2026-07-07)')


[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40
[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [46]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [47]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6738, 1180), features: 985


In [48]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [49]:
# =============================================================================
# FEATURES D'AMPLITUDE — calculées directement sur VIX et SPX
# Objectif : capturer l'ACCÉLÉRATION et l'ERRATICITÉ du VIX, pas seulement
# la direction. Ce sont les features manquantes pour prédire UP_FORT.
# =============================================================================

def add_amplitude_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des features spécifiques à l'intensité du mouvement VIX."""
    df = df.copy()

    # Colonne VIX brute
    vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX"
    spx_col = "SP500_Price" if "SP500_Price" in df.columns else None
    vix = df[vix_col]

    # 1. Volatilité-de-la-volatilité : vol réalisée du VIX sur 5 et 10 jours
    df["vix_vol_of_vol_5d"]  = vix.pct_change().rolling(5,  min_periods=3).std()
    df["vix_vol_of_vol_10d"] = vix.pct_change().rolling(10, min_periods=5).std()

    # 2. Momentum à court terme : accélération récente du VIX
    vix_ret1 = vix.pct_change(1)
    df["vix_momentum_2d"] = vix.pct_change(2)
    df["vix_momentum_3d"] = vix.pct_change(3)

    # 3. Accélération : changement de vitesse (dérivée seconde)
    df["vix_acceleration_1d"] = vix_ret1 - vix_ret1.shift(1)
    df["vix_acceleration_3d"] = vix_ret1 - vix_ret1.shift(3)

    # 4. Sur-extension court terme : distance au MA5 et MA10
    ma5  = vix.rolling(5,  min_periods=3).mean()
    ma10 = vix.rolling(10, min_periods=5).mean()
    df["vix_vs_ma5"]  = (vix - ma5)  / ma5.replace(0, np.nan)
    df["vix_vs_ma10"] = (vix - ma10) / ma10.replace(0, np.nan)

    # 5. Z-score court terme du VIX (5 et 10 jours)
    df["vix_zscore_5d"]  = (vix - ma5)  / vix.rolling(5,  min_periods=3).std().replace(0, np.nan)
    df["vix_zscore_10d"] = (vix - ma10) / vix.rolling(10, min_periods=5).std().replace(0, np.nan)

    # 6. Persistance du stress : % des 10 derniers jours avec VIX > MA20
    ma20 = vix.rolling(20, min_periods=10).mean()
    above_ma20 = (vix > ma20).astype(float)
    df["vix_pct_above_ma20_10d"] = above_ma20.rolling(10, min_periods=5).mean()

    # 7. Erraticité : max des |rendements| sur 5 jours vs moyenne
    abs_ret = vix.pct_change().abs()
    df["vix_max_abs_ret_5d"]  = abs_ret.rolling(5, min_periods=3).max()
    df["vix_mean_abs_ret_5d"] = abs_ret.rolling(5, min_periods=3).mean()
    df["vix_erratic_ratio"]   = df["vix_max_abs_ret_5d"] / df["vix_mean_abs_ret_5d"].replace(0, np.nan)

    # 8. Ratio vol_5d / vol_60d (spike court terme vs bruit de fond)
    df["vix_vol_ratio_5_60"] = (
        vix.pct_change().rolling(5,  min_periods=3).std() /
        vix.pct_change().rolling(60, min_periods=30).std().replace(0, np.nan)
    )

    # 9. SPX amplitude features (si disponible)
    if spx_col and spx_col in df.columns:
        spx = df[spx_col]
        spx_ret = spx.pct_change()
        df["spx_vol_5d"]         = spx_ret.rolling(5,  min_periods=3).std()
        df["spx_momentum_3d"]    = spx.pct_change(3)
        df["spx_abs_ret_max_5d"] = spx_ret.abs().rolling(5, min_periods=3).max()

    df = df.replace([np.inf, -np.inf], np.nan)
    n_added = sum(1 for c in df.columns if c in [
        "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
        "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
        "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
        "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
        "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
    ])
    print(f"[AMPLITUDE FEATURES] {n_added} features d'amplitude ajoutées")
    return df

df_post_features = add_amplitude_features(df_post_features)

# Mettre à jour la liste de features
amplitude_feat_names = [
    "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
    "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
    "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
    "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
    "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
]
new_feats = [f for f in amplitude_feat_names if f in df_post_features.columns]
features_post_engineering = features_post_engineering + new_feats
print(f"[AMPLITUDE FEATURES] features_post_engineering : {len(features_post_engineering)} total ({len(new_feats)} nouvelles)")


[AMPLITUDE FEATURES] 18 features d'amplitude ajoutées
[AMPLITUDE FEATURES] features_post_engineering : 1003 total (18 nouvelles)


In [50]:
# Ancienne version AmplitudeTargetBuilder (3 quantiles) — neutralisée.
# La version conditionnelle par régime est en cellule 24.


In [51]:
def clf_configs():
    """
    Hyperparamètres ajustés pour détecter les extrêmes (UP_FORT / DOWN_FORT).
    Changements vs run précédent :
    - XGBoost : arbres plus profonds (max_depth 3-5), plus d'estimateurs (125-200)
    - LightGBM : num_leaves plus élevé (15-31), min_child_samples réduit
    - GradientBoosting : subsample ajouté pour robustesse
    - LogisticRegression : C plus large pour moins de régularisation
    """
    return {
        "XGBoost": (XGBClassifier,
            {"max_depth":[3,5],"learning_rate":[0.03,0.05],
             "n_estimators":[125,200],"subsample":[0.8],"colsample_bytree":[0.7,0.8],
             "min_child_weight":[1,3]},
            {"random_state":RANDOM_STATE,"eval_metric":"mlogloss",
             "objective":"multi:softprob","n_jobs":-1}),
        "LightGBM": (LGBMClassifier,
            {"num_leaves":[15,31],"learning_rate":[0.03,0.05],
             "n_estimators":[125,200],"max_depth":[4,6],
             "min_child_samples":[5,10]},
            {"random_state":RANDOM_STATE,"verbose":-1,"class_weight":"balanced"}),
        "GradientBoosting": (GradientBoostingClassifier,
            {"n_estimators":[125,200],"learning_rate":[0.03,0.05],
             "max_depth":[3,5],"min_samples_leaf":[5,10],"subsample":[0.8]},
            {"random_state":RANDOM_STATE}),
        "RandomForest": (RandomForestClassifier,
            {"n_estimators":[200],"max_depth":[5,8],"min_samples_leaf":[5,10]},
            {"random_state":RANDOM_STATE,"n_jobs":-1,"class_weight":"balanced"}),
        "LogisticRegression": (LogisticRegression,
            {"C":[0.1,1.0,10.0]},
            {"random_state":RANDOM_STATE,"max_iter":2000,
             "class_weight":"balanced","multi_class":"multinomial","solver":"lbfgs"}),
    }


In [52]:
def generate_all_interactions(df: pd.DataFrame, base_features: list,
                               top_n: int = 20, rolling_w: int = 20,
                               eps: float = 1e-8) -> pd.DataFrame:
    feats = [f for f in base_features[:top_n] if f in df.columns]
    n = len(feats)
    new_cols = {}
    for i in range(n):
        for j in range(i+1, n):
            fi, fj = feats[i], feats[j]
            si, sj = df[fi], df[fj]
            safe_denom = sj.where(sj.abs() >= eps, np.nan)
            new_cols[f"{fi}__div__{fj}"]      = si / safe_denom
            new_cols[f"{fi}__minus__{fj}"]    = si - sj
            new_cols[f"{fi}__prod__{fj}"]     = si * sj
            diff = si - sj
            roll_std = diff.rolling(rolling_w, min_periods=rolling_w//2).std()
            new_cols[f"{fi}__zrel__{fj}"]     = diff / roll_std.replace(0, np.nan)
            ma_i = si.rolling(rolling_w, min_periods=rolling_w//2).mean()
            ma_j = sj.rolling(rolling_w, min_periods=rolling_w//2).mean()
            new_cols[f"{fi}__macross__{fj}"]  = ma_i / ma_j.where(ma_j.abs() >= eps, np.nan)
            new_cols[f"{fi}__ret5x__{fj}"]    = si.pct_change(5) * sj
    idf = pd.DataFrame(new_cols, index=df.index).replace([np.inf,-np.inf], np.nan)
    idf = idf.dropna(axis=1, how="all")
    print(f"[INTERACTIONS] {len(feats)} features → {idf.shape[1]} colonnes")
    return idf


In [53]:
def shap_select_features(X_train: pd.DataFrame, y_train: np.ndarray,
                          top_n: int, task: str = "clf", label: str = "") -> list:
    """SHAP sur XGBoost pilote. task='clf' ou 'reg'."""
    if task == "clf":
        pilot = XGBClassifier(n_estimators=SHAP_PILOT_N_ESTIMATORS,
                              max_depth=3, learning_rate=0.05, subsample=0.8,
                              eval_metric="mlogloss", objective="multi:softprob",
                              random_state=RANDOM_STATE, n_jobs=-1)
    else:
        pilot = XGBRegressor(n_estimators=SHAP_PILOT_N_ESTIMATORS,
                             max_depth=3, learning_rate=0.05, subsample=0.8,
                             random_state=RANDOM_STATE, n_jobs=-1)

    # LabelEncoder pour la classification XGBoost
    if task == "clf":
        le = LabelEncoder()
        y_enc = le.fit_transform(y_train)
        pilot.fit(X_train.values, y_enc)
    else:
        pilot.fit(X_train.values, y_train)

    explainer   = shap.TreeExplainer(pilot)
    shap_values = explainer.shap_values(X_train.values)

    # shap_values peut être :
    #   - list de arrays 2D (ancienne API SHAP, classification multi-classe)
    #   - array 3D (n_samples, n_features, n_classes) — nouvelle API SHAP
    #   - array 2D (n_samples, n_features) — binaire ou régression
    if isinstance(shap_values, list):
        # list[class_k] = (n_samples, n_features) → moyenne sur classes
        shap_arr = np.mean([np.abs(sv) for sv in shap_values], axis=0)
    elif shap_values.ndim == 3:
        # (n_samples, n_features, n_classes) → moyenne sur classes (axis=2)
        shap_arr = np.abs(shap_values).mean(axis=2)
    else:
        shap_arr = np.abs(shap_values)

    # shap_arr est maintenant (n_samples, n_features)
    mean_abs = pd.Series(shap_arr.mean(axis=0), index=X_train.columns)
    top = mean_abs.sort_values(ascending=False).head(top_n).index.tolist()
    if label:
        print(f"[SHAP {label}] top-{top_n} sur {len(X_train.columns)} candidates")
    return top


def run_phase1_shap_amplitude(horizon_days: int, task: str):
    """
    Phase 1 SHAP pour une tâche donnée (clf ou reg) et un horizon.
    Retourne {regime: {features, train_start, n_interactions}}
    """
    print(f"\n{'#'*80}")
    print(f"# PHASE 1 SHAP | task={task.upper()} | h={horizon_days}j")
    print(f"{'#'*80}")

    atb = AmplitudeTargetBuilder(horizon_days=horizon_days)
    df_full = atb.build(df_post_features, train_end=TEST_DATE)
    df_train = df_full.loc[df_full.index < pd.Timestamp(TEST_DATE)].copy()
    feats_avail = [f for f in features_post_engineering if f in df_train.columns]

    target_col = "VIX_Amplitude_Class" if task == "clf" else "VIX_Return"
    result = {}

    for regime in ALL_REGIMES:
        print(f"\n--- {regime} ---")
        df_tr = df_train if regime == "GLOBAL" else df_train.loc[df_train["VIX_Regime"] == regime]

        if len(df_tr) < 80 or df_tr[target_col].nunique() < 2:
            print(f"[WARN] {regime}: échantillon insuffisant. Skip.")
            continue

        cleaner = TrainFittedCleaner()
        X_b = pd.DataFrame(cleaner.fit_transform(df_tr[feats_avail]),
                           columns=feats_avail, index=df_tr.index).dropna(axis=1, how="all")
        X_b_sc = pd.DataFrame(StandardScaler().fit_transform(X_b),
                               columns=X_b.columns, index=X_b.index)
        y_b = df_tr[target_col].loc[X_b_sc.index].values

        # Étape A : SHAP base → top-40
        top_base = shap_select_features(X_b_sc, y_b, SHAP_TOP_BASE_N, task, f"{regime} base")

        # Étape B : interactions sur top-20
        idf = generate_all_interactions(df_tr[top_base], top_base,
                                         N_TOP_FOR_INTERACTIONS, INTERACTION_ROLLING_WINDOW)
        extended = top_base + list(idf.columns)
        df_ext = pd.concat([df_tr[top_base], idf], axis=1).loc[:, ~pd.concat([df_tr[top_base],idf],axis=1).columns.duplicated()]

        cleaner2 = TrainFittedCleaner()
        X_e = pd.DataFrame(cleaner2.fit_transform(df_ext[extended]),
                           columns=extended, index=df_ext.index).dropna(axis=1, how="all")
        X_e_sc = pd.DataFrame(StandardScaler().fit_transform(X_e),
                               columns=X_e.columns, index=X_e.index)
        y_e = df_tr[target_col].loc[X_e_sc.index].values

        # Étape C : SHAP (base + interactions) → top-30 final
        top_final = shap_select_features(X_e_sc, y_e, SHAP_TOP_FINAL_N, task,
                                          f"{regime} base+inter")
        n_inter = sum(1 for f in top_final if any(s in f for s in
                      ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]))
        print(f"[FINAL] {regime} h={horizon_days}j task={task}: "
              f"{len(top_final)} features ({n_inter} interactions)")

        # Étape D : TRAIN_START auto
        _SEPS = ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]
        base_in_df   = [f for f in top_final if f in df_full.columns]
        inter_needed = [f for f in top_final if f not in df_full.columns]
        src_set = set(base_in_df)
        for fname in inter_needed:
            for sep in _SEPS:
                if sep in fname:
                    a, b = fname.split(sep, 1)
                    if a in df_full.columns: src_set.add(a)
                    if b in df_full.columns: src_set.add(b)
                    break

        src_list = [f for f in src_set if f in df_full.columns]
        if inter_needed and src_list:
            idf_all = generate_all_interactions(df_full[src_list], src_list,
                                                 len(src_list), INTERACTION_ROLLING_WINDOW)
            avail_inter = [f for f in inter_needed if f in idf_all.columns]
            df_start = pd.concat([df_full[base_in_df], idf_all[avail_inter]], axis=1)
        else:
            df_start = df_full[base_in_df]

        first_valid = df_start.dropna(how="any").index.min()
        train_start = (first_valid if not pd.isna(first_valid)
                       else pd.Timestamp(TRAIN_START)).strftime("%Y-%m-%d")
        print(f"[TRAIN_START] {regime} h={horizon_days}j task={task}: {train_start}")

        result[regime] = {
            "features":       top_final,
            "train_start":    train_start,
            "n_interactions": n_inter,
            "class_thresholds": atb.class_thresholds_by_regime_,
        }

    return result


# Exécution Phase 1 pour les 2 horizons × 2 tâches
phase1 = {}
for h in HORIZONS_TO_TEST:
    phase1[h] = {}
    for task in ["clf", "reg"]:
        phase1[h][task] = run_phase1_shap_amplitude(h, task)

# Sauvegarde Phase 1
p1_rows = []
for h in HORIZONS_TO_TEST:
    for task in ["clf","reg"]:
        for regime, info in phase1[h][task].items():
            p1_rows.append({"Horizon":h,"Task":task,"Regime":regime,
                            "Train_Start":info["train_start"],
                            "N_Features":len(info["features"]),
                            "N_Interactions":info["n_interactions"],
                            "Features":json.dumps(info["features"])})
p1_df = pd.DataFrame(p1_rows)
p1_df.to_csv(OUTPUT_DIR / "phase1_shap_features.csv", index=False)
print(f"\n[SAVE] phase1_shap_features.csv")
display(p1_df[["Horizon","Task","Regime","Train_Start","N_Features","N_Interactions"]])



################################################################################
# PHASE 1 SHAP | task=CLF | h=5j
################################################################################
[TARGET h=5j] 0 jours flat supprimés, 6733 lignes restantes
  [CALM] q25=-4.429%, q75=8.274% (n=1835)
  [NORMAL] q25=-8.527%, q75=7.104% (n=1891)
  [STRESS] q25=-10.117%, q75=5.730% (n=1839)
  [GLOBAL] q25=-7.769%, q75=7.263%
VIX_Amplitude_Class
DOWN_FAIBLE    1912
DOWN_FORT      1677
UP_FAIBLE      1454
UP_FORT        1690
Name: count, dtype: int64

--- CALM ---
[SHAP CALM base] top-40 sur 1003 candidates
[INTERACTIONS] 20 features → 1140 colonnes
[SHAP CALM base+inter] top-30 sur 1180 candidates
[FINAL] CALM h=5j task=clf: 30 features (17 interactions)
[INTERACTIONS] 28 features → 2268 colonnes
[TRAIN_START] CALM h=5j task=clf: 2000-11-15

--- NORMAL ---
[SHAP NORMAL base] top-40 sur 1003 candidates
[INTERACTIONS] 20 features → 1140 colonnes
[SHAP NORMAL base+inter] top-30 sur 1180 candidate

,Horizon,Task,Regime,Train_Start,N_Features,N_Interactions
0,5,clf,CALM,2000-11-15,30,17
1,5,clf,NORMAL,2000-11-02,30,21
2,5,clf,STRESS,2000-11-15,30,25
3,5,clf,GLOBAL,2000-11-15,30,23
4,5,reg,CALM,2000-11-15,30,25
5,5,reg,NORMAL,2001-08-14,30,28
6,5,reg,STRESS,2000-11-15,30,25
7,5,reg,GLOBAL,2001-08-14,30,24


In [54]:
class AmplitudeTargetBuilder:
    """
    Cible amplitude avec quantiles q25/q75 calculés PAR RÉGIME sur le train.
    Jours flat (|rendement| < FLAT_THRESHOLD_ABS) supprimés définitivement.
    """
    def __init__(self, horizon_days: int = 5):
        self.horizon_days  = horizon_days
        self.vix_col       = None
        self.calm_thr_     = None
        self.stress_thr_   = None
        # seuils par régime : {regime: (q25, q75)}
        self.class_thresholds_by_regime_ = {}

    def build(self, df: pd.DataFrame, train_end: str) -> pd.DataFrame:
        df = df.copy().sort_index()
        for c in ["VIX_Price", "VIX", "^VIX"]:
            if c in df.columns:
                self.vix_col = c
                break
        if self.vix_col is None:
            raise ValueError("Colonne VIX introuvable.")

        vix = pd.to_numeric(df[self.vix_col], errors="coerce")
        vix_train = vix.loc[vix.index < pd.Timestamp(train_end)]
        self.calm_thr_   = vix_train.quantile(0.33)
        self.stress_thr_ = vix_train.quantile(0.67)

        regime = pd.Series("NORMAL", index=df.index)
        regime.loc[vix < self.calm_thr_]    = "CALM"
        regime.loc[vix >= self.stress_thr_] = "STRESS"

        future_vix = vix.shift(-self.horizon_days)
        vix_return = (future_vix / vix) - 1

        flat_mask = vix_return.abs() < FLAT_THRESHOLD_ABS
        n_flat = flat_mask.sum()

        df["VIX_Return"] = vix_return
        df["VIX_Regime"] = regime
        df = df.loc[~flat_mask].dropna(subset=["VIX_Return"])

        print(f"[TARGET h={self.horizon_days}j] {n_flat} jours flat supprimés, {len(df)} lignes restantes")

        # ── Quantiles CONDITIONNELS au régime (calculés sur train uniquement) ──
        df_train = df.loc[df.index < pd.Timestamp(train_end)]

        def classify_regime(row):
            r = row["VIX_Regime"]
            ret = row["VIX_Return"]
            q25, q75 = self.class_thresholds_by_regime_.get(r, (0, 0))
            if ret < q25:   return "DOWN_FORT"
            if ret < 0:     return "DOWN_FAIBLE"
            if ret < q75:   return "UP_FAIBLE"
            return "UP_FORT"

        for reg in ["CALM", "NORMAL", "STRESS"]:
            sub = df_train.loc[df_train["VIX_Regime"] == reg, "VIX_Return"]
            if len(sub) < 20:
                q25, q75 = sub.quantile(0.25) if len(sub) else 0, sub.quantile(0.75) if len(sub) else 0
            else:
                q25, q75 = sub.quantile(CLASS_QUANTILES[0]), sub.quantile(CLASS_QUANTILES[1])
            self.class_thresholds_by_regime_[reg] = (q25, q75)
            print(f"  [{reg}] q25={q25:.3%}, q75={q75:.3%} (n={len(sub)})")

        # GLOBAL utilise les quantiles globaux
        q25_g = df_train["VIX_Return"].quantile(CLASS_QUANTILES[0])
        q75_g = df_train["VIX_Return"].quantile(CLASS_QUANTILES[1])
        self.class_thresholds_by_regime_["GLOBAL"] = (q25_g, q75_g)
        print(f"  [GLOBAL] q25={q25_g:.3%}, q75={q75_g:.3%}")

        # Note : classification utilise 0 comme frontière UP/DOWN,
        # et q25/q75 comme frontière FORT/FAIBLE — les classes FAIBLE
        # peuvent être déséquilibrées selon les régimes (voulu : c'est
        # la réalité économique, pas un artefact)
        df["VIX_Amplitude_Class"] = df.apply(classify_regime, axis=1)

        print(df["VIX_Amplitude_Class"].value_counts().sort_index())
        return df


In [55]:
def clf_configs():
    return {
        "XGBoost": (XGBClassifier,
            {"max_depth":[2,3],"learning_rate":[0.03,0.05],
             "n_estimators":[75,125],"subsample":[0.8],"colsample_bytree":[0.8]},
            {"random_state":RANDOM_STATE,"eval_metric":"mlogloss",
             "objective":"multi:softprob","n_jobs":-1}),
        "LightGBM": (LGBMClassifier,
            {"num_leaves":[7,15],"learning_rate":[0.03,0.05],
             "n_estimators":[75,125],"max_depth":[3,5]},
            {"random_state":RANDOM_STATE,"verbose":-1,"class_weight":"balanced"}),
        "GradientBoosting": (GradientBoostingClassifier,
            {"n_estimators":[75,125],"learning_rate":[0.03,0.05],
             "max_depth":[2,3],"min_samples_leaf":[10]},
            {"random_state":RANDOM_STATE}),
        "RandomForest": (RandomForestClassifier,
            {"n_estimators":[150],"max_depth":[3,5],"min_samples_leaf":[10,20]},
            {"random_state":RANDOM_STATE,"n_jobs":-1,"class_weight":"balanced"}),
        "LogisticRegression": (LogisticRegression,
            {"C":[0.01,0.1,1.0]},
            {"random_state":RANDOM_STATE,"max_iter":2000,
             "class_weight":"balanced","multi_class":"multinomial","solver":"lbfgs"}),
    }

def compute_hierarchical_metrics(y_true, y_pred):
    """
    Métriques à 3 niveaux :
    1. UP vs DOWN (binaire agrégé) — F1, Precision, Recall
    2. FORT vs FAIBLE au sein de UP — F1
    3. FORT vs FAIBLE au sein de DOWN — F1
    """
    # Mapping vers direction binaire
    dir_map = {
        "DOWN_FORT":  "DOWN", "DOWN_FAIBLE": "DOWN",
        "UP_FAIBLE":  "UP",   "UP_FORT":     "UP"
    }
    y_dir_true = [dir_map.get(y, "DOWN") for y in y_true]
    y_dir_pred = [dir_map.get(y, "DOWN") for y in y_pred]

    # Niveau 1 : direction
    acc_dir   = accuracy_score(y_dir_true, y_dir_pred)
    f1_up     = f1_score(y_dir_true, y_dir_pred, pos_label="UP",   average="binary", zero_division=0)
    f1_down   = f1_score(y_dir_true, y_dir_pred, pos_label="DOWN", average="binary", zero_division=0)
    prec_up   = precision_score(y_dir_true, y_dir_pred, pos_label="UP",   average="binary", zero_division=0)
    rec_up    = recall_score(y_dir_true, y_dir_pred, pos_label="UP",     average="binary", zero_division=0)
    prec_down = precision_score(y_dir_true, y_dir_pred, pos_label="DOWN", average="binary", zero_division=0)
    rec_down  = recall_score(y_dir_true, y_dir_pred, pos_label="DOWN",   average="binary", zero_division=0)
    f1_dir_macro = (f1_up + f1_down) / 2

    # Niveau 2 : FORT vs FAIBLE au sein de UP
    up_idx = [i for i, y in enumerate(y_true) if dir_map.get(y) == "UP"]
    if len(up_idx) > 10:
        yt_up = ["FORT" if y_true[i] == "UP_FORT" else "FAIBLE" for i in up_idx]
        yp_up = ["FORT" if y_pred[i] == "UP_FORT" else "FAIBLE" for i in up_idx]
        f1_up_fort   = f1_score(yt_up, yp_up, pos_label="FORT",   average="binary", zero_division=0)
        f1_up_faible = f1_score(yt_up, yp_up, pos_label="FAIBLE", average="binary", zero_division=0)
        acc_up_sub   = accuracy_score(yt_up, yp_up)
    else:
        f1_up_fort = f1_up_faible = acc_up_sub = np.nan

    # Niveau 3 : FORT vs FAIBLE au sein de DOWN
    dn_idx = [i for i, y in enumerate(y_true) if dir_map.get(y) == "DOWN"]
    if len(dn_idx) > 10:
        yt_dn = ["FORT" if y_true[i] == "DOWN_FORT" else "FAIBLE" for i in dn_idx]
        yp_dn = ["FORT" if y_pred[i] == "DOWN_FORT" else "FAIBLE" for i in dn_idx]
        f1_dn_fort   = f1_score(yt_dn, yp_dn, pos_label="FORT",   average="binary", zero_division=0)
        f1_dn_faible = f1_score(yt_dn, yp_dn, pos_label="FAIBLE", average="binary", zero_division=0)
        acc_dn_sub   = accuracy_score(yt_dn, yp_dn)
    else:
        f1_dn_fort = f1_dn_faible = acc_dn_sub = np.nan

    # Niveau 0 : 4 classes brutes
    f1_4cls = f1_score(y_true, y_pred, average="macro",
                        labels=CLASS_LABELS, zero_division=0)

    return {
        # Niveau 0 : 4 classes
        "F1_4cls_macro":   round(f1_4cls, 4),
        "Acc_4cls":        round(accuracy_score(y_true, y_pred), 4),
        # Niveau 1 : direction
        "Acc_Direction":   round(acc_dir, 4),
        "F1_Dir_macro":    round(f1_dir_macro, 4),
        "F1_UP":           round(f1_up, 4),
        "Prec_UP":         round(prec_up, 4),
        "Rec_UP":          round(rec_up, 4),
        "F1_DOWN":         round(f1_down, 4),
        "Prec_DOWN":       round(prec_down, 4),
        "Rec_DOWN":        round(rec_down, 4),
        # Niveau 2 : amplitude dans UP
        "Acc_UP_sub":      round(acc_up_sub, 4) if not np.isnan(acc_up_sub) else None,
        "F1_UP_FORT":      round(f1_up_fort, 4) if not np.isnan(f1_up_fort) else None,
        "F1_UP_FAIBLE":    round(f1_up_faible, 4) if not np.isnan(f1_up_faible) else None,
        # Niveau 3 : amplitude dans DOWN
        "Acc_DOWN_sub":    round(acc_dn_sub, 4) if not np.isnan(acc_dn_sub) else None,
        "F1_DOWN_FORT":    round(f1_dn_fort, 4) if not np.isnan(f1_dn_fort) else None,
        "F1_DOWN_FAIBLE":  round(f1_dn_faible, 4) if not np.isnan(f1_dn_faible) else None,
    }


In [56]:
def generate_all_interactions(df, base_features, top_n=20, rolling_w=20, eps=1e-8):
    feats = [f for f in base_features[:top_n] if f in df.columns]
    new_cols = {}
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            fi, fj = feats[i], feats[j]
            si, sj = df[fi], df[fj]
            safe_denom = sj.where(sj.abs() >= eps, np.nan)
            new_cols[f"{fi}__div__{fj}"]     = si / safe_denom
            new_cols[f"{fi}__minus__{fj}"]   = si - sj
            new_cols[f"{fi}__prod__{fj}"]    = si * sj
            diff = si - sj
            rs = diff.rolling(rolling_w, min_periods=rolling_w//2).std()
            new_cols[f"{fi}__zrel__{fj}"]    = diff / rs.replace(0, np.nan)
            ma_i = si.rolling(rolling_w, min_periods=rolling_w//2).mean()
            ma_j = sj.rolling(rolling_w, min_periods=rolling_w//2).mean()
            new_cols[f"{fi}__macross__{fj}"] = ma_i / ma_j.where(ma_j.abs() >= eps, np.nan)
            new_cols[f"{fi}__ret5x__{fj}"]   = si.pct_change(5) * sj
    idf = pd.DataFrame(new_cols, index=df.index).replace([np.inf,-np.inf], np.nan)
    idf = idf.dropna(axis=1, how="all")
    print(f"[INTER] {len(feats)} features → {idf.shape[1]} colonnes")
    return idf


def shap_select_features(X_train, y_train, top_n, label=""):
    pilot = XGBClassifier(n_estimators=SHAP_PILOT_N_ESTIMATORS,
                          max_depth=3, learning_rate=0.05, subsample=0.8,
                          eval_metric="mlogloss", objective="multi:softprob",
                          random_state=RANDOM_STATE, n_jobs=-1)
    le = LabelEncoder(); le.fit(CLASS_LABELS)
    pilot.fit(X_train.values, le.transform(y_train))
    expl = shap.TreeExplainer(pilot)
    sv = expl.shap_values(X_train.values)
    if isinstance(sv, list):
        arr = np.mean([np.abs(s) for s in sv], axis=0)
    elif np.array(sv).ndim == 3:
        arr = np.abs(sv).mean(axis=2)
    else:
        arr = np.abs(sv)
    scores = pd.Series(arr.mean(axis=0), index=X_train.columns)
    top = scores.sort_values(ascending=False).head(top_n).index.tolist()
    if label: print(f"[SHAP {label}] top-{top_n}/{len(X_train.columns)}")
    return top


print("="*70)
print("PHASE 1 SHAP — h=5j, classification 4 classes, quantiles conditionnels")
print("="*70)

atb = AmplitudeTargetBuilder(horizon_days=5)
df_full = atb.build(df_post_features, train_end=TEST_DATE)
df_train_full = df_full.loc[df_full.index < pd.Timestamp(TEST_DATE)].copy()
feats_avail = [f for f in features_post_engineering if f in df_train_full.columns]

phase1 = {}
_SEPS = ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]

for regime in ALL_REGIMES:
    print(f"\n--- {regime} ---")
    df_tr = df_train_full if regime == "GLOBAL" else df_train_full[df_train_full["VIX_Regime"]==regime]
    if len(df_tr) < 80 or df_tr["VIX_Amplitude_Class"].nunique() < 2:
        print("[WARN] Skip."); continue

    cleaner = TrainFittedCleaner()
    X_b = pd.DataFrame(cleaner.fit_transform(df_tr[feats_avail]),
                        columns=feats_avail, index=df_tr.index).dropna(axis=1, how="all")
    X_b_sc = pd.DataFrame(StandardScaler().fit_transform(X_b),
                           columns=X_b.columns, index=X_b.index)
    y_b = df_tr["VIX_Amplitude_Class"].loc[X_b_sc.index].values

    top_base = shap_select_features(X_b_sc, y_b, SHAP_TOP_BASE_N, f"{regime} base")
    idf = generate_all_interactions(df_tr[top_base], top_base, N_TOP_FOR_INTERACTIONS, INTERACTION_ROLLING_WINDOW)
    extended = top_base + list(idf.columns)
    df_ext = pd.concat([df_tr[top_base], idf], axis=1).loc[:, ~pd.concat([df_tr[top_base],idf],axis=1).columns.duplicated()]

    cleaner2 = TrainFittedCleaner()
    X_e = pd.DataFrame(cleaner2.fit_transform(df_ext[extended]),
                        columns=extended, index=df_ext.index).dropna(axis=1, how="all")
    X_e_sc = pd.DataFrame(StandardScaler().fit_transform(X_e),
                           columns=X_e.columns, index=X_e.index)
    y_e = df_tr["VIX_Amplitude_Class"].loc[X_e_sc.index].values

    top_final = shap_select_features(X_e_sc, y_e, SHAP_TOP_FINAL_N, f"{regime} +inter")
    n_inter = sum(1 for f in top_final if any(s in f for s in _SEPS))
    print(f"[FINAL] {regime}: {len(top_final)} features ({n_inter} interactions)")

    # TRAIN_START auto
    base_in = [f for f in top_final if f in df_full.columns]
    inter_n = [f for f in top_final if f not in df_full.columns]
    src_set = set(base_in)
    for fname in inter_n:
        for sep in _SEPS:
            if sep in fname:
                a, b = fname.split(sep, 1)
                if a in df_full.columns: src_set.add(a)
                if b in df_full.columns: src_set.add(b)
                break
    src_list = [f for f in src_set if f in df_full.columns]
    if inter_n and src_list:
        idf_all = generate_all_interactions(df_full[src_list], src_list, len(src_list), INTERACTION_ROLLING_WINDOW)
        avail = [f for f in inter_n if f in idf_all.columns]
        df_start = pd.concat([df_full[base_in], idf_all[avail]], axis=1)
    else:
        df_start = df_full[base_in]

    first_valid = df_start.dropna(how="any").index.min()
    train_start = (first_valid if not pd.isna(first_valid) else pd.Timestamp(TRAIN_START)).strftime("%Y-%m-%d")
    print(f"[TRAIN_START] {regime}: {train_start}")

    phase1[regime] = {"features": top_final, "train_start": train_start, "n_interactions": n_inter}

pd.DataFrame([{"Regime":r,"Train_Start":v["train_start"],"N_Features":len(v["features"]),"N_Interactions":v["n_interactions"]}
               for r, v in phase1.items()]).to_csv(OUTPUT_DIR/"phase1_shap.csv", index=False)
print("\n[SAVE] phase1_shap.csv")


PHASE 1 SHAP — h=5j, classification 4 classes, quantiles conditionnels
[TARGET h=5j] 0 jours flat supprimés, 6733 lignes restantes
  [CALM] q25=-4.429%, q75=8.274% (n=1835)
  [NORMAL] q25=-8.527%, q75=7.104% (n=1891)
  [STRESS] q25=-10.117%, q75=5.730% (n=1839)
  [GLOBAL] q25=-7.769%, q75=7.263%
VIX_Amplitude_Class
DOWN_FAIBLE    1912
DOWN_FORT      1677
UP_FAIBLE      1454
UP_FORT        1690
Name: count, dtype: int64

--- CALM ---
[SHAP CALM base] top-40/1003
[INTER] 20 features → 1140 colonnes
[SHAP CALM +inter] top-30/1180
[FINAL] CALM: 30 features (17 interactions)
[INTER] 28 features → 2268 colonnes
[TRAIN_START] CALM: 2000-11-15

--- NORMAL ---
[SHAP NORMAL base] top-40/1003
[INTER] 20 features → 1140 colonnes
[SHAP NORMAL +inter] top-30/1180
[FINAL] NORMAL: 30 features (21 interactions)
[INTER] 26 features → 1950 colonnes
[TRAIN_START] NORMAL: 2000-11-02

--- STRESS ---
[SHAP STRESS base] top-40/1003
[INTER] 20 features → 1140 colonnes
[SHAP STRESS +inter] top-30/1180
[FINAL] S

In [ ]:
print("="*70)
print("PHASE 2 — h=5j, métriques hiérarchiques 3 niveaux")
print("="*70)

clf_rows = []
model_num = 0

# Reconstruire le df pour h=5 (atb déjà initialisé en Phase 1)
for regime in ALL_REGIMES:
    info = phase1.get(regime)
    if not info: continue

    top_final   = info["features"]
    train_start = info["train_start"]
    print(f"\n{'='*50}\n{regime} | {len(top_final)} features | train_start={train_start}")

    df_w = df_full.loc[df_full.index >= pd.Timestamp(train_start)].copy()

    _SEPS = ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]
    base_in = [f for f in top_final if f in df_w.columns]
    inter_n = [f for f in top_final if f not in df_w.columns]
    if inter_n:
        src = set(base_in)
        for fname in inter_n:
            for sep in _SEPS:
                if sep in fname:
                    a, b = fname.split(sep, 1)
                    if a in df_w.columns: src.add(a)
                    if b in df_w.columns: src.add(b)
                    break
        src_l = [f for f in src if f in df_w.columns]
        if src_l:
            idf2 = generate_all_interactions(df_w[src_l], src_l, len(src_l), INTERACTION_ROLLING_WINDOW)
            avail = [f for f in inter_n if f in idf2.columns]
            df_w = pd.concat([df_w, idf2[avail]], axis=1)

    feats_run = [f for f in top_final if f in df_w.columns]
    df_w[feats_run] = df_w[feats_run].replace([np.inf,-np.inf], np.nan)

    df_tr_full = df_w.loc[df_w.index < pd.Timestamp(TEST_DATE)]
    df_te_full = df_w.loc[df_w.index >= pd.Timestamp(TEST_DATE)]

    mask_tr = pd.Series(True, index=df_tr_full.index) if regime=="GLOBAL" else df_tr_full["VIX_Regime"]==regime
    mask_te = pd.Series(True, index=df_te_full.index) if regime=="GLOBAL" else df_te_full["VIX_Regime"]==regime

    dtr, dte = df_tr_full.loc[mask_tr], df_te_full.loc[mask_te]
    if len(dtr) < 80 or len(dte) < 20: print("[WARN] trop petit. Skip."); continue

    y_tr_raw = dtr["VIX_Amplitude_Class"].values
    y_te     = dte["VIX_Amplitude_Class"].values
    if len(np.unique(y_tr_raw)) < 2: print("[WARN] 1 classe. Skip."); continue

    print(f"  Train: {len(dtr)} | Test: {len(dte)}")
    print("  Distribution test:", pd.Series(y_te).value_counts().to_dict())

    cleaner = TrainFittedCleaner()
    X_tr_c = cleaner.fit_transform(dtr[feats_run])
    X_te_c = cleaner.transform(dte[feats_run])
    sc = StandardScaler()
    X_tr_s = pd.DataFrame(sc.fit_transform(X_tr_c), columns=feats_run, index=dtr.index)
    X_te_s = pd.DataFrame(sc.transform(X_te_c),     columns=feats_run, index=dte.index)

    le = LabelEncoder(); le.fit(CLASS_LABELS)
    y_tr_enc = le.transform(y_tr_raw)
    try:
        sm = SMOTE(random_state=RANDOM_STATE)
        X_tr_arr, y_tr_sm = sm.fit_resample(X_tr_s.values, y_tr_enc)
        X_tr_s = pd.DataFrame(X_tr_arr, columns=feats_run)
        y_tr_fit = y_tr_sm
    except Exception:
        y_tr_fit = y_tr_enc

    max_n = min(MAX_N_FEATURES, len(feats_run))
    cv = TimeSeriesSplit(n_splits=3)

    for algo_name, (Cls, pgrid, fixed) in clf_configs().items():
        for n in range(MIN_N_FEATURES, max_n+1):
            fn = feats_run[:n]
            Xtr = X_tr_s[fn].values if isinstance(X_tr_s, pd.DataFrame) else X_tr_s[:,:n]
            Xte = X_te_s[fn].values if isinstance(X_te_s, pd.DataFrame) else X_te_s[:,:n]
            try:
                grid = GridSearchCV(Cls(**fixed), pgrid, scoring="f1_macro", cv=cv, n_jobs=-1)
                grid.fit(Xtr, y_tr_fit)
                model = grid.best_estimator_
            except Exception:
                continue

            y_pred_enc = model.predict(Xte)
            y_pred_str = le.inverse_transform(y_pred_enc)
            y_te_str   = list(y_te)

            m = compute_hierarchical_metrics(y_te_str, list(y_pred_str))
            model_num += 1
            clf_rows.append({"Model_Number":model_num,"Regime":regime,"Algo":algo_name,
                             "N_Features":n,"Train_Start":train_start,"Features":json.dumps(fn), **m})

        print(f"  {algo_name} ✓")

clf_df = pd.DataFrame(clf_rows)
clf_df.to_csv(OUTPUT_DIR/"phase2_clf_hierarchical.csv", index=False)
print(f"\n[DONE] {len(clf_df)} modèles évalués")


PHASE 2 — h=5j, métriques hiérarchiques 3 niveaux

CALM | 30 features | train_start=2000-11-15
[INTER] 28 features → 2268 colonnes
  Train: 1835 | Test: 255
  Distribution test: {'UP_FAIBLE': 88, 'UP_FORT': 66, 'DOWN_FORT': 56, 'DOWN_FAIBLE': 45}
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

NORMAL | 30 features | train_start=2000-11-02
[INTER] 26 features → 1950 colonnes
  Train: 1857 | Test: 570
  Distribution test: {'DOWN_FAIBLE': 171, 'UP_FORT': 164, 'DOWN_FORT': 121, 'UP_FAIBLE': 114}
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

STRESS | 30 features | train_start=2000-11-15
[INTER] 23 features → 1518 colonnes
  Train: 1805 | Test: 343
  Distribution test: {'DOWN_FAIBLE': 131, 'DOWN_FORT': 108, 'UP_FORT': 68, 'UP_FAIBLE': 36}
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

GLOBAL | 30 features | train_start=2000-11-15
[INTER] 26 features → 1950 colonnes
  Train:

In [ ]:
print("="*70)
print("SYNTHÈSE — h=5j | Métriques hiérarchiques")
print("="*70)
print("Seuil de référence : direction binaire > 0.5 (hasard=0.5)")
print("                     4 classes > 0.25 (hasard=0.25)\n")

best_rows = []

for regime in ALL_REGIMES:
    sub = clf_df[clf_df["Regime"]==regime]
    if sub.empty: continue

    # Sélection par F1_Dir_macro (métrique niveau 1 — direction)
    best = sub.loc[sub["F1_Dir_macro"].idxmax()]
    flag_dir  = "✓" if best["F1_Dir_macro"]  > 0.50 else "✗"
    flag_4cls = "✓" if best["F1_4cls_macro"] > 0.25 else "✗"

    print(f"\n{'─'*60}")
    print(f"  {regime} | {best['Algo']} N={best['N_Features']} | train_start={best['Train_Start']}")
    print(f"  ── Niveau 0 : 4 classes ──")
    print(f"    F1_macro  = {best['F1_4cls_macro']:.4f}  {flag_4cls}  |  Acc = {best['Acc_4cls']:.4f}")
    print(f"  ── Niveau 1 : UP vs DOWN ──")
    print(f"    Acc_dir   = {best['Acc_Direction']:.4f}  |  F1_dir = {best['F1_Dir_macro']:.4f}  {flag_dir}")
    print(f"    UP   → F1={best['F1_UP']:.4f}  P={best['Prec_UP']:.4f}  R={best['Rec_UP']:.4f}")
    print(f"    DOWN → F1={best['F1_DOWN']:.4f}  P={best['Prec_DOWN']:.4f}  R={best['Rec_DOWN']:.4f}")
    print(f"  ── Niveau 2 : amplitude dans UP ──")
    print(f"    F1_UP_FORT={best['F1_UP_FORT']}  F1_UP_FAIBLE={best['F1_UP_FAIBLE']}  Acc={best['Acc_UP_sub']}")
    print(f"  ── Niveau 3 : amplitude dans DOWN ──")
    print(f"    F1_DOWN_FORT={best['F1_DOWN_FORT']}  F1_DOWN_FAIBLE={best['F1_DOWN_FAIBLE']}  Acc={best['Acc_DOWN_sub']}")

    best_rows.append({**{"Regime":regime}, **best.to_dict()})

best_df = pd.DataFrame(best_rows)

excel_path = OUTPUT_DIR / "vix_amplitude_h5_hierarchical_report.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    pd.read_csv(OUTPUT_DIR/"phase1_shap.csv").to_excel(writer,"Phase1_SHAP",index=False)
    clf_df.to_excel(writer,"All_Results",index=False)
    best_df.to_excel(writer,"Best_Per_Regime",index=False)

print(f"\n[SAVE] {excel_path}")
print("\n[NOTE] Aucun modèle enregistré — validation explicite requise.")


SYNTHÈSE — h=5j | Métriques hiérarchiques
Seuil de référence : direction binaire > 0.5 (hasard=0.5)
                     4 classes > 0.25 (hasard=0.25)


────────────────────────────────────────────────────────────
  CALM | LightGBM N=6 | train_start=2000-11-15
  ── Niveau 0 : 4 classes ──
    F1_macro  = 0.2936  ✓  |  Acc = 0.3020
  ── Niveau 1 : UP vs DOWN ──
    Acc_dir   = 0.5843  |  F1_dir = 0.5829  ✓
    UP   → F1=0.5583  P=0.7791  R=0.4351
    DOWN → F1=0.6074  P=0.4852  R=0.8119
  ── Niveau 2 : amplitude dans UP ──
    F1_UP_FORT=0.3689  F1_UP_FAIBLE=0.6829  Acc=0.5779
  ── Niveau 3 : amplitude dans DOWN ──
    F1_DOWN_FORT=0.5714  F1_DOWN_FAIBLE=0.4667  Acc=0.5248

────────────────────────────────────────────────────────────
  NORMAL | LogisticRegression N=6 | train_start=2000-11-02
  ── Niveau 0 : 4 classes ──
    F1_macro  = 0.3107  ✓  |  Acc = 0.3123
  ── Niveau 1 : UP vs DOWN ──
    Acc_dir   = 0.5509  |  F1_dir = 0.5491  ✓
    UP   → F1=0.5776  P=0.5335  R=0.6295
    DOWN